# Phase 4: Data Augmentation & Hybrid Retraining

This notebook retrains the classifier on a balanced dataset (Original + Synthetic).

**Goals:**
1. Combine `train_defect` (original) and `synthetic_defect` (GAN-generated).
2. Mix with `train_normal`.
3. Retrain ResNet18.
4. Compare performance with Phase 2 Baseline.

**Improvements:**
- Stronger data augmentation (rotation, color jitter)
- Learning rate scheduler to avoid overshooting
- More training epochs (15)

In [ ]:
# 1. Imports & Setup
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Dataset
import os
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 15  # More epochs for the larger augmented dataset

In [ ]:
# 2. Dataset Logic (Hybrid)
class HybridDataset(Dataset):
    def __init__(self, root_dir, synthetic_dir, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        # Load Normal (0)
        normal_dir = os.path.join(root_dir, 'train_normal')
        self._load_from_folder(normal_dir, 0)
        
        # Load Real Defect (1)
        defect_dir = os.path.join(root_dir, 'train_defect')
        self._load_from_folder(defect_dir, 1)
        
        # Load Synthetic Defect (1)
        if os.path.exists(synthetic_dir):
            self._load_from_folder(synthetic_dir, 1)
        else:
            print(f"Warning: Synthetic directory {synthetic_dir} not found!")
            
    def _load_from_folder(self, folder, label):
        if not os.path.exists(folder): 
            return
        count = 0
        for f in os.listdir(folder):
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                self.image_paths.append(os.path.join(folder, f))
                self.labels.append(label)
                count += 1
        print(f"Loaded {count} images from {folder} (Label {label})")

    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)

# Transforms with stronger augmentation
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]),
    'val': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]),
}

# Create Datasets
print("Creating Hybrid Training Set...")
train_dataset = HybridDataset(root_dir='processed_data', 
                              synthetic_dir='synthetic_defect',
                              transform=data_transforms['train'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Validation
class ValDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        # Load Valid Mixed
        self._load_from_folder(os.path.join(root_dir, 'valid_mixed', 'normal'), 0)
        self._load_from_folder(os.path.join(root_dir, 'valid_mixed', 'defect'), 1)
            
    def _load_from_folder(self, folder, label):
        if not os.path.exists(folder): return
        for f in os.listdir(folder):
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                self.image_paths.append(os.path.join(folder, f))
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)

val_dataset = ValDataset('processed_data', transform=data_transforms['val'])
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

# Print class balance
n_normal = sum(1 for l in train_dataset.labels if l == 0)
n_defect = sum(1 for l in train_dataset.labels if l == 1)
print(f"Training set balance - Normal: {n_normal}, Defect: {n_defect}")

In [ ]:
# 3. Model & Training
def get_model():
    model = models.resnet18(pretrained=True)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 1)
    return model.to(device)

model = get_model()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

def train_model(model, train_loader, val_loader, epochs=EPOCHS):
    best_recall = 0.0 # Optimization target: Recall (Labels are 1 for Defect)
    train_losses = []
    val_recalls = []
    val_f1s = []
    
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_loss)
            
        # Validate
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device).unsqueeze(1)
                outputs = model(inputs)
                preds = torch.sigmoid(outputs) > 0.5
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        recall = recall_score(all_labels, all_preds, zero_division=0)
        precision = precision_score(all_labels, all_preds, zero_division=0)
        f1 = f1_score(all_labels, all_preds, zero_division=0)
        val_recalls.append(recall)
        val_f1s.append(f1)
        
        print(f"Loss: {epoch_loss:.4f} | Recall: {recall:.4f} | Precision: {precision:.4f} | F1: {f1:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")
        
        if recall > best_recall:
            best_recall = recall
            torch.save(model.state_dict(), 'best_hybrid_model.pth')
            print(f"  -> Saved best model (Recall: {best_recall:.4f})")
        
        # Step the scheduler
        scheduler.step()
            
    # Plot training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(train_losses)
    ax1.set_title('Training Loss')
    ax1.set_xlabel('Epoch')
    ax2.plot(val_recalls, label='Recall')
    ax2.plot(val_f1s, label='F1')
    ax2.set_title('Validation Metrics')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    plt.tight_layout()
    plt.show()
    
    return model

train_model(model, train_loader, val_loader)

In [ ]:
# 4. Comparative Evaluation on Test Set

def evaluate(model_path, loader, name="Model"):
    model = get_model()
    # Handle missing file if not run sequentially
    if not os.path.exists(model_path):
        print(f"{model_path} not found. Skipping.")
        return {}
    
    model.load_state_dict(torch.load(model_path))
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds = torch.sigmoid(outputs) > 0.5
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    metrics = {
        'Accuracy': accuracy_score(all_labels, all_preds),
        'Precision': precision_score(all_labels, all_preds, zero_division=0),
        'Recall': recall_score(all_labels, all_preds, zero_division=0),
        'F1': f1_score(all_labels, all_preds, zero_division=0)
    }
    print(f"--- {name} Results ---")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
    return metrics

# Load Test Data
test_dataset = ValDataset('processed_data') # Reusing ValDataset logic but point to test_mixed manually
test_dataset.image_paths = []
test_dataset.labels = []
test_dataset._load_from_folder('processed_data/test_mixed/normal', 0)
test_dataset._load_from_folder('processed_data/test_mixed/defect', 1)
print(f"Test size: {len(test_dataset)}")
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Compare
baseline_metrics = evaluate('best_baseline_model.pth', test_loader, "Baseline (Phase 2)")
hybrid_metrics = evaluate('best_hybrid_model.pth', test_loader, "Hybrid (Phase 4)")

# Visualization
if baseline_metrics and hybrid_metrics:
    labels = list(baseline_metrics.keys())
    baseline_vals = list(baseline_metrics.values())
    hybrid_vals = list(hybrid_metrics.values())
    
    x = np.arange(len(labels))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline', color='#ff7f7f')
    rects2 = ax.bar(x + width/2, hybrid_vals, width, label='Hybrid (GAN Augmented)', color='#7fbf7f')
    
    ax.set_ylabel('Scores')
    ax.set_title('Performance Comparison: Baseline vs Hybrid')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    ax.set_ylim(0, 1.1)
    
    # Add value labels on bars
    for rect in rects1:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}', xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
    for rect in rects2:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}', xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()